In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pickle

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    load_mat_file,

    preprocess_nir_uco_cube,
    build_minimal_nir_uco_object_database,

    make_reference_image, 
    make_binary_mask, 
    clean_mask, 
    label_objects_with_watershed,

    plot_hypercube_band_slider,
    plot_image2d,
    plot_distribution_with_curve,
    plot_label_overlay_from_image_db,
    plot_object_spectra,
    plot_spectral_distribution,
    plot_object_view,
    plot_object_grid,
    plot_object_areas,
    plot_counts_by_group,
    plot_spectra,
    plot_lines_from_dataframe,
    plot_bar_values,
    plot_explained_variance,
    plot_scores,
    build_scores_dataframe,
    sample_scores_dataframe,
    plot_scores_density,
    plot_scores_distribution,
    summarize_scores_by_object,
    plot_object_score_summary,
    plot_loadings,
    plot_biplot,
    plot_pca_diagnostic,
    plot_pca_metric_t2,
    plot_pca_metric_q,
    plot_simca_distance,
    plot_simca_rule_metric,
    plot_decision_counts,
    plot_object_decision_map,
    plot_pca_metric_heatmap,
    plot_pca_metric_tradeoff,
    plot_pca_metric_ranking,
    
    object_db_to_matrix,
    SpectralPreprocessor,

    PCAModel,

    compare_pca_representations,
    binary_class_separation_scores,
    compute_pca_summary_metrics,
    add_pca_selection_score,

    SIMCAClassifier,
    SimpleSIMCARule,
    AltSIMCARule,
    CombinedIndexSIMCARule,
    DataDrivenSIMCARule,
)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# PARAMETERS
PROCESSED_DATA_DIR = PROJECT_ROOT / "HSI Data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

START_NM = 889
END_NM = 1702
ORIGINAL_BANDS = 89
N_REMOVE = 6

DATA_MODE = "reflectance"

In [4]:
processed_dir = Path(PROCESSED_DATA_DIR)

with open(processed_dir / "nir_uco_minimal_object_db_reflectance.pkl", "rb") as f:
    object_db = pickle.load(f)

with open(processed_dir / "nir_uco_minimal_image_db_reflectance.pkl", "rb") as f:
    image_db = pickle.load(f)


In [5]:
raw_wavelengths = np.linspace(START_NM, END_NM, ORIGINAL_BANDS)
wavelengths = raw_wavelengths[N_REMOVE:]

In [6]:
# modelling parameters
train_batches = [1, 2]
projection_batches = [3, 4]

target_classes = ["almond", "peanut"]

train_filters = {
    "sample_kind": ["pure"],
    "object_nut_type": target_classes,
    "batch": train_batches,
}

projection_filters = {
    "sample_kind": ["pure"],
    "object_nut_type": target_classes,
    "batch": projection_batches,
}

# From object to matrix

In [7]:
X_train_mean_raw, y_train_mean, meta_train_mean = object_db_to_matrix(object_db, level="object", spectrum_field="mean_spectrum", filters=train_filters)
X_projection_mean_raw, y_projection_mean, meta_projection_mean = object_db_to_matrix(object_db, level="object", spectrum_field="mean_spectrum", filters=projection_filters)

X_train_pixel_raw, y_train_pixel, meta_train_pixel = object_db_to_matrix(object_db, level="pixel", m=40, filters=train_filters)
X_projection_pixel_raw, y_projection_pixel, meta_projection_pixel = object_db_to_matrix(object_db, level="pixel", m=40, filters=projection_filters)

X_train_balanced_pixel_raw, y_train_balanced_pixel, meta_train_balanced_pixel = object_db_to_matrix(object_db, level="balanced_pixel", m=40, filters=train_filters)
X_projection_balanced_pixel_raw, y_projection_balanced_pixel, meta_projection_balanced_pixel = object_db_to_matrix(object_db, level="balanced_pixel", m=40, filters=projection_filters)

# Pre-processing

In [8]:
preprocessing_configs = {
    "raw": ("raw",),
    "absorbance": ("absorbance",),
    "snv": ("snv",),
    "msc": ("msc",),
    "1st_derivative": ("sg_d1",),
    "2nd_derivative": ("sg_d2",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_msc": ("absorbance", "msc"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_sg_d2": ("absorbance", "sg_d2"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "absorbance_snv_sg_d2": ("absorbance", "snv", "sg_d2"),
}

## mean

In [9]:
# Apply all preprocessing configurations to training and projection data
preprocessed_data_mean = {}

for name, steps in preprocessing_configs.items():
    print(f"Applying preprocessing: {name} | steps={steps}")
    
    preprocessor = SpectralPreprocessor(
        steps=steps,
        sg_window_length=9,
        sg_polyorder=2,
    )
    
    X_train_mean_pre = preprocessor.fit_transform(
        X_train_mean_raw,
        wavelengths=wavelengths,
    )
    
    X_projection_mean_pre = preprocessor.transform(
        X_projection_mean_raw,
    )
    
    preprocessed_data_mean[name] = {
        "steps": steps,
        "preprocessor": preprocessor,
        "X_train": X_train_mean_pre,
        "X_projection": X_projection_mean_pre,
        "y_train": y_train_mean,
        "y_projection": y_projection_mean,
        "meta_train": meta_train_mean,
        "meta_projection": meta_projection_mean,
    }

print("\nDone.")

Applying preprocessing: raw | steps=('raw',)
Applying preprocessing: absorbance | steps=('absorbance',)
Applying preprocessing: snv | steps=('snv',)
Applying preprocessing: msc | steps=('msc',)
Applying preprocessing: 1st_derivative | steps=('sg_d1',)
Applying preprocessing: 2nd_derivative | steps=('sg_d2',)
Applying preprocessing: absorbance_snv | steps=('absorbance', 'snv')
Applying preprocessing: absorbance_msc | steps=('absorbance', 'msc')
Applying preprocessing: absorbance_sg_d1 | steps=('absorbance', 'sg_d1')
Applying preprocessing: absorbance_sg_d2 | steps=('absorbance', 'sg_d2')
Applying preprocessing: absorbance_snv_sg_d1 | steps=('absorbance', 'snv', 'sg_d1')
Applying preprocessing: absorbance_snv_sg_d2 | steps=('absorbance', 'snv', 'sg_d2')

Done.


In [10]:
for name, item in preprocessed_data_mean.items():
    X_combined_mean = np.vstack([
        item["X_train"],
        item["X_projection"],
    ])
    
    subset_labels = np.array(
        ["train"] * item["X_train"].shape[0]
        + ["projection"] * item["X_projection"].shape[0]
    )
    
    class_labels = np.concatenate([
        item["y_train"],
        item["y_projection"],
    ])
    
    combined_labels = np.array([
        f"{subset} | {label}"
        for subset, label in zip(subset_labels, class_labels)
    ])
    
    plot_spectra(
        X_combined_mean,
        wavelengths=wavelengths,
        labels=combined_labels,
        reducer="mean",
        title=f"Train vs projection mean spectra: {name}",
        y_title="Preprocessed intensity",
    )

In [11]:
class_difference_rows_mean = []

for name, item in preprocessed_data_mean.items():
    Xtr = item["X_train"]
    ytr = np.asarray(item["y_train"])
    
    if not {"almond", "peanut"}.issubset(set(ytr)):
        continue
    
    mean_almond = Xtr[ytr == "almond"].mean(axis=0)
    mean_peanut = Xtr[ytr == "peanut"].mean(axis=0)
    diff = mean_peanut - mean_almond
    
    class_difference_rows_mean.append({
        "preprocessing": name,
        "mean_abs_difference": float(np.mean(np.abs(diff))),
        "max_abs_difference": float(np.max(np.abs(diff))),
        "l2_difference": float(np.linalg.norm(diff)),
    })

class_difference_df_mean = pd.DataFrame(class_difference_rows_mean)
display(class_difference_df_mean.sort_values("l2_difference", ascending=False))

plot_bar_values(
    class_difference_df_mean["preprocessing"],
    class_difference_df_mean["l2_difference"],
    title="Class mean difference after preprocessing",
    x_title="Preprocessing",
    y_title="L2 distance between peanut and almond mean spectra",
)

,preprocessing,mean_abs_difference,max_abs_difference,l2_difference
6,absorbance_snv,0.045687,0.204333,0.539916
2,snv,0.042007,0.179266,0.473368
1,absorbance,0.046752,0.079330,0.390445
0,raw,0.033235,0.048163,0.266894
7,absorbance_msc,0.005387,0.024218,0.063706
3,msc,0.003619,0.015524,0.040807
10,absorbance_snv_sg_d1,0.001034,0.006092,0.012938
8,absorbance_sg_d1,0.000169,0.000625,0.001814
4,1st_derivative,0.000075,0.000393,0.000876
11,absorbance_snv_sg_d2,0.000040,0.000274,0.000558


In [12]:
diff_spectra_mean = []
diff_labels_mean = []

for name, item in preprocessed_data_mean.items():
    Xtr = item["X_train"]
    ytr = np.asarray(item["y_train"])
    
    if not {"almond", "peanut"}.issubset(set(ytr)):
        continue
    
    mean_almond = Xtr[ytr == "almond"].mean(axis=0)
    mean_peanut = Xtr[ytr == "peanut"].mean(axis=0)
    diff = mean_peanut - mean_almond
    
    diff_spectra_mean.append(diff)
    diff_labels_mean.append(name)

diff_spectra_mean = np.vstack(diff_spectra_mean)

plot_spectra(
    diff_spectra_mean,
    wavelengths=wavelengths,
    labels=np.array(diff_labels_mean),
    reducer="none",
    title="Peanut mean spectrum - almond mean spectrum after preprocessing",
    y_title="Difference",
)

## all pixels

In [13]:
# Apply all preprocessing configurations to training and projection data
preprocessed_data_pixel = {}

for name, steps in preprocessing_configs.items():
    print(f"Applying preprocessing: {name} | steps={steps}")
    
    preprocessor = SpectralPreprocessor(
        steps=steps,
        sg_window_length=9,
        sg_polyorder=2,
    )
    
    X_train_pixel_pre = preprocessor.fit_transform(
        X_train_pixel_raw,
        wavelengths=wavelengths,
    )
    
    X_projection_pixel_pre = preprocessor.transform(
        X_projection_pixel_raw,
    )
    
    preprocessed_data_pixel[name] = {
        "steps": steps,
        "preprocessor": preprocessor,
        "X_train": X_train_pixel_pre,
        "X_projection": X_projection_pixel_pre,
        "y_train": y_train_pixel,
        "y_projection": y_projection_pixel,
        "meta_train": meta_train_pixel,
        "meta_projection": meta_projection_pixel,
    }

print("\nDone.")

Applying preprocessing: raw | steps=('raw',)
Applying preprocessing: absorbance | steps=('absorbance',)
Applying preprocessing: snv | steps=('snv',)
Applying preprocessing: msc | steps=('msc',)
Applying preprocessing: 1st_derivative | steps=('sg_d1',)
Applying preprocessing: 2nd_derivative | steps=('sg_d2',)
Applying preprocessing: absorbance_snv | steps=('absorbance', 'snv')
Applying preprocessing: absorbance_msc | steps=('absorbance', 'msc')
Applying preprocessing: absorbance_sg_d1 | steps=('absorbance', 'sg_d1')
Applying preprocessing: absorbance_sg_d2 | steps=('absorbance', 'sg_d2')
Applying preprocessing: absorbance_snv_sg_d1 | steps=('absorbance', 'snv', 'sg_d1')
Applying preprocessing: absorbance_snv_sg_d2 | steps=('absorbance', 'snv', 'sg_d2')

Done.


In [14]:
for name, item in preprocessed_data_pixel.items():
    X_combined_pixel = np.vstack([
        item["X_train"],
        item["X_projection"],
    ])
    
    subset_labels = np.array(
        ["train"] * item["X_train"].shape[0]
        + ["projection"] * item["X_projection"].shape[0]
    )
    
    class_labels = np.concatenate([
        item["y_train"],
        item["y_projection"],
    ])
    
    combined_labels = np.array([
        f"{subset} | {label}"
        for subset, label in zip(subset_labels, class_labels)
    ])
    
    plot_spectra(
        X_combined_pixel,
        wavelengths=wavelengths,
        labels=combined_labels,
        reducer="mean",
        title=f"Train vs projection mean spectra: {name}",
        y_title="Preprocessed intensity",
    )

In [15]:
class_difference_rows_pixel = []

for name, item in preprocessed_data_pixel.items():
    Xtr = item["X_train"]
    ytr = np.asarray(item["y_train"])
    
    if not {"almond", "peanut"}.issubset(set(ytr)):
        continue
    
    mean_almond = Xtr[ytr == "almond"].mean(axis=0)
    mean_peanut = Xtr[ytr == "peanut"].mean(axis=0)
    diff = mean_peanut - mean_almond
    
    class_difference_rows_pixel.append({
        "preprocessing": name,
        "mean_abs_difference": float(np.mean(np.abs(diff))),
        "max_abs_difference": float(np.max(np.abs(diff))),
        "l2_difference": float(np.linalg.norm(diff)),
    })

class_difference_df_pixel = pd.DataFrame(class_difference_rows_pixel)
display(class_difference_df_pixel.sort_values("l2_difference", ascending=False))

plot_bar_values(
    class_difference_df_pixel["preprocessing"],
    class_difference_df_pixel["l2_difference"],
    title="Class mean difference after preprocessing",
    x_title="Preprocessing",
    y_title="L2 distance between peanut and almond mean spectra",
)

,preprocessing,mean_abs_difference,max_abs_difference,l2_difference
6,absorbance_snv,0.043445,0.180313,0.514071
2,snv,0.040034,0.166463,0.451860
1,absorbance,0.033915,0.065973,0.294468
0,raw,0.030071,0.044608,0.241942
7,absorbance_msc,0.005522,0.023032,0.066127
3,msc,0.004227,0.020036,0.044668
10,absorbance_snv_sg_d1,0.000973,0.005021,0.011732
8,absorbance_sg_d1,0.000188,0.000884,0.002166
4,1st_derivative,0.000073,0.000390,0.000869
11,absorbance_snv_sg_d2,0.000036,0.000211,0.000474


In [16]:
diff_spectra_pixel = []
diff_labels_pixel = []

for name, item in preprocessed_data_pixel.items():
    Xtr = item["X_train"]
    ytr = np.asarray(item["y_train"])
    
    if not {"almond", "peanut"}.issubset(set(ytr)):
        continue
    
    mean_almond = Xtr[ytr == "almond"].mean(axis=0)
    mean_peanut = Xtr[ytr == "peanut"].mean(axis=0)
    diff = mean_peanut - mean_almond
    
    diff_spectra_pixel.append(diff)
    diff_labels_pixel.append(name)

diff_spectra_pixel = np.vstack(diff_spectra_pixel)

plot_spectra(
    diff_spectra_pixel,
    wavelengths=wavelengths,
    labels=np.array(diff_labels_pixel),
    reducer="none",
    title="Peanut mean spectrum - almond mean spectrum after preprocessing",
    y_title="Difference",
)

## balanced pixels

In [17]:
# Apply all preprocessing configurations to training and projection data
preprocessed_data_balanced_pixel = {}

for name, steps in preprocessing_configs.items():
    print(f"Applying preprocessing: {name} | steps={steps}")
    
    preprocessor = SpectralPreprocessor(
        steps=steps,
        sg_window_length=9,
        sg_polyorder=2,
    )
    
    X_train_balanced_pixel_pre = preprocessor.fit_transform(
        X_train_balanced_pixel_raw,
        wavelengths=wavelengths,
    )
    
    X_projection_balanced_pixel_pre = preprocessor.transform(
        X_projection_balanced_pixel_raw,
    )
    
    preprocessed_data_balanced_pixel[name] = {
        "steps": steps,
        "preprocessor": preprocessor,
        "X_train": X_train_balanced_pixel_pre,
        "X_projection": X_projection_balanced_pixel_pre,
        "y_train": y_train_balanced_pixel,
        "y_projection": y_projection_balanced_pixel,
        "meta_train": meta_train_balanced_pixel,
        "meta_projection": meta_projection_balanced_pixel,
    }

print("\nDone.")

Applying preprocessing: raw | steps=('raw',)
Applying preprocessing: absorbance | steps=('absorbance',)
Applying preprocessing: snv | steps=('snv',)
Applying preprocessing: msc | steps=('msc',)
Applying preprocessing: 1st_derivative | steps=('sg_d1',)
Applying preprocessing: 2nd_derivative | steps=('sg_d2',)
Applying preprocessing: absorbance_snv | steps=('absorbance', 'snv')
Applying preprocessing: absorbance_msc | steps=('absorbance', 'msc')
Applying preprocessing: absorbance_sg_d1 | steps=('absorbance', 'sg_d1')
Applying preprocessing: absorbance_sg_d2 | steps=('absorbance', 'sg_d2')
Applying preprocessing: absorbance_snv_sg_d1 | steps=('absorbance', 'snv', 'sg_d1')
Applying preprocessing: absorbance_snv_sg_d2 | steps=('absorbance', 'snv', 'sg_d2')

Done.


In [18]:
for name, item in preprocessed_data_balanced_pixel.items():
    X_combined_balanced_pixel = np.vstack([
        item["X_train"],
        item["X_projection"],
    ])
    
    subset_labels = np.array(
        ["train"] * item["X_train"].shape[0]
        + ["projection"] * item["X_projection"].shape[0]
    )
    
    class_labels = np.concatenate([
        item["y_train"],
        item["y_projection"],
    ])
    
    combined_labels = np.array([
        f"{subset} | {label}"
        for subset, label in zip(subset_labels, class_labels)
    ])
    
    plot_spectra(
        X_combined_balanced_pixel,
        wavelengths=wavelengths,
        labels=combined_labels,
        reducer="mean",
        title=f"Train vs projection mean spectra: {name}",
        y_title="Preprocessed intensity",
    )

In [19]:
class_difference_rows_balanced_pixel = []

for name, item in preprocessed_data_balanced_pixel.items():
    Xtr = item["X_train"]
    ytr = np.asarray(item["y_train"])
    
    if not {"almond", "peanut"}.issubset(set(ytr)):
        continue
    
    mean_almond = Xtr[ytr == "almond"].mean(axis=0)
    mean_peanut = Xtr[ytr == "peanut"].mean(axis=0)
    diff = mean_peanut - mean_almond
    
    class_difference_rows_balanced_pixel.append({
        "preprocessing": name,
        "mean_abs_difference": float(np.mean(np.abs(diff))),
        "max_abs_difference": float(np.max(np.abs(diff))),
        "l2_difference": float(np.linalg.norm(diff)),
    })

class_difference_df_balanced_pixel = pd.DataFrame(class_difference_rows_balanced_pixel)
display(class_difference_df_balanced_pixel.sort_values("l2_difference", ascending=False))

plot_bar_values(
    class_difference_df_balanced_pixel["preprocessing"],
    class_difference_df_balanced_pixel["l2_difference"],
    title="Class mean difference after preprocessing",
    x_title="Preprocessing",
    y_title="L2 distance between peanut and almond mean spectra",
)

,preprocessing,mean_abs_difference,max_abs_difference,l2_difference
6,absorbance_snv,0.044490,0.184850,0.523429
2,snv,0.041215,0.171589,0.467723
1,absorbance,0.038435,0.072730,0.331159
0,raw,0.035668,0.050142,0.285918
7,absorbance_msc,0.005829,0.023515,0.069614
3,msc,0.005258,0.025715,0.054620
10,absorbance_snv_sg_d1,0.000965,0.004806,0.011603
8,absorbance_sg_d1,0.000210,0.001068,0.002436
4,1st_derivative,0.000073,0.000414,0.000888
11,absorbance_snv_sg_d2,0.000035,0.000197,0.000457


In [20]:
diff_spectra_balanced_pixel = []
diff_labels_balanced_pixel = []

for name, item in preprocessed_data_balanced_pixel.items():
    Xtr = item["X_train"]
    ytr = np.asarray(item["y_train"])
    
    if not {"almond", "peanut"}.issubset(set(ytr)):
        continue
    
    mean_almond = Xtr[ytr == "almond"].mean(axis=0)
    mean_peanut = Xtr[ytr == "peanut"].mean(axis=0)
    diff = mean_peanut - mean_almond
    
    diff_spectra_balanced_pixel.append(diff)
    diff_labels_balanced_pixel.append(name)

diff_spectra_balanced_pixel = np.vstack(diff_spectra_balanced_pixel)

plot_spectra(
    diff_spectra_balanced_pixel,
    wavelengths=wavelengths,
    labels=np.array(diff_labels_balanced_pixel),
    reducer="none",
    title="Peanut mean spectrum - almond mean spectrum after preprocessing",
    y_title="Difference",
)

# PCA

In [21]:
n_components = 10
n_components_diagnostic = 4

In [22]:
def build_pca_metrics_dataframe(
    pca_results,
    preprocessed_data,
    matrix_method,
    candidate_preprocessings=None,
    n_components_metrics=3,
):
    """
    Build a PCA comparison table for one matrix representation.

    Parameters
    ----------
    pca_results : dict
        Dictionary containing fitted PCA results.
    preprocessed_data : dict
        Dictionary containing X_train, X_projection, y_train, y_projection and metadata.
    matrix_method : str
        One of: "object_mean", "object_median", "all_pixels", "balanced_pixels".
    candidate_preprocessings : list or None
        If provided, restrict metrics to these preprocessings.
    n_components_metrics : int
        Number of PCA components used to compute score-space metrics.

    Returns
    -------
    df : pandas.DataFrame
        PCA comparison metrics.
    """
    rows = []

    if candidate_preprocessings is None:
        candidate_preprocessings = list(pca_results.keys())

    for name in candidate_preprocessings:
        res = pca_results[name]
        item = preprocessed_data[name]

        pca_model = res["pca"]

        meta_train = (
            pd.DataFrame(item["meta_train"])
            if isinstance(item["meta_train"], dict)
            else item["meta_train"].copy()
        )

        meta_projection = (
            pd.DataFrame(item["meta_projection"])
            if isinstance(item["meta_projection"], dict)
            else item["meta_projection"].copy()
        )

        metrics = compute_pca_summary_metrics(
            pca_model=pca_model,
            X_train=item["X_train"],
            T_train=res["T_train"],
            y_train=item["y_train"],
            metadata_train=meta_train.to_dict(orient="list"),
            X_projection=item["X_projection"],
            T_projection=res["T_projection"],
            y_projection=item["y_projection"],
            metadata_projection=meta_projection.to_dict(orient="list"),
            n_components=n_components_metrics,
            matrix_method=matrix_method,
        )

        sep = res.get("separation", {})

        row = {
            "matrix_method": matrix_method,
            "preprocessing": name,
            "n_train": len(item["y_train"]),
            "n_projection": len(item["y_projection"]),

            # Existing quick diagnostics
            "centroid_distance_pc1_pc2": sep.get("centroid_distance_pc1_pc2", np.nan),
            "fisher_pc1": sep.get("fisher_pc1", np.nan),
            "fisher_pc2": sep.get("fisher_pc2", np.nan),
            "fisher_pc3": sep.get("fisher_pc3", np.nan),
            "mahalanobis_pc1_pc2": sep.get("mahalanobis_pc1_pc2", np.nan),
            "mahalanobis_pc1_pc2_pc3": sep.get("mahalanobis_pc1_pc2_pc3", np.nan),

            # New metrics
            **metrics,
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
candidate_preprocessings = [
    'snv',
    'absorbance_snv',
    'absorbance_sg_d1',
    'msc',
    'absorbance_msc',
    'raw',
    'absorbance',
]

In [23]:
candidate_preprocessings = list(preprocessed_data_mean.keys())

## mean

In [24]:
pca_results_mean = {}

for key, item in preprocessed_data_mean.items():
    Xtr = item["X_train"]
    Xpr = item["X_projection"]
    ytr = item["y_train"]
    ypr = item["y_projection"]
    
    pca_model = PCAModel(
        n_components=n_components,
        center=True,
    )
    
    pca_model.fit(Xtr)
    
    Ttr = pca_model.transform(Xtr)
    Tpr = pca_model.transform(Xpr)
    
    # Reconstruction with the first n_components_diagnostic components
    Tpr_A = Tpr[:, :n_components_diagnostic]
    P_A = pca_model.loadings_[:, :n_components_diagnostic]
    Xpr_hat = Tpr_A @ P_A.T + pca_model.mean_
    
    rmse_pr = np.sqrt(np.mean((Xpr - Xpr_hat) ** 2, axis=1))
    
    # Class separation scores on training scores
    sep = binary_class_separation_scores(
        Ttr,
        ytr,
        n_components=n_components_diagnostic,
    )
    
    pca_results_mean[key] = {
        "pca": pca_model,
        "T_train": Ttr,
        "T_projection": Tpr,
        "rmse_projection": rmse_pr,
        "separation": sep,
        "explained_variance_ratio": pca_model.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": pca_model.cumulative_explained_variance_ratio_,
    }

print("PCA models fitted for:")
for name in pca_results_mean:
    print("-", name)

PCA models fitted for:
- raw
- absorbance
- snv
- msc
- 1st_derivative
- 2nd_derivative
- absorbance_snv
- absorbance_msc
- absorbance_sg_d1
- absorbance_sg_d2
- absorbance_snv_sg_d1
- absorbance_snv_sg_d2


In [25]:
pca_comparison_df_mean = build_pca_metrics_dataframe(
    pca_results=pca_results_mean,
    preprocessed_data=preprocessed_data_mean,
    matrix_method="object_mean",
    candidate_preprocessings=list(pca_results_mean.keys()),
    n_components_metrics=3,
)
pca_comparison_df_mean["projection_q_deviation"] = np.abs(
    np.log(pca_comparison_df_mean["projection_train_q_ratio"].astype(float))
)

pca_comparison_df_mean_scored = add_pca_selection_score(
    pca_comparison_df_mean,
    profile="object",
    group_col=None,
    score_col="selection_score",
)

display(pca_comparison_df_mean_scored.sort_values("selection_score", ascending=False)[[
    "preprocessing", "selection_score", "selection_flag", "class_trace_ratio", "mahalanobis_pc1_pc2_pc3", "batch_trace_ratio", "class_over_batch_ratio",
    "mean_train_projection_shift_norm", "max_train_projection_shift_norm", "projection_q_deviation", "train_q_q95", "projection_q_q95",
    "train_t2_q95", "projection_t2_q95", "cum_pc3", "ncomp_95"
]])


,preprocessing,selection_score,selection_flag,class_trace_ratio,mahalanobis_pc1_pc2_pc3,batch_trace_ratio,class_over_batch_ratio,mean_train_projection_shift_norm,max_train_projection_shift_norm,projection_q_deviation,train_q_q95,projection_q_q95,train_t2_q95,projection_t2_q95,cum_pc3,ncomp_95
9,absorbance_sg_d2,2.759304,weak_class_separation,0.055106,0.057983,0.001499,36.755729,0.309187,0.326372,0.025490,1.194252e-09,1.072608e-09,8.182101,7.900635,0.974954,2
7,absorbance_msc,2.273695,candidate,0.480243,6.627051,0.002677,179.402809,0.493531,0.784386,0.244209,1.122945e-03,1.230456e-03,6.654247,7.950378,0.899133,5
6,absorbance_snv,2.273591,candidate,0.481741,6.681698,0.002691,179.048812,0.493864,0.785219,0.244553,8.106347e-02,8.835828e-02,6.646744,7.944596,0.898381,5
2,snv,2.191771,candidate,0.460763,7.332611,0.001716,268.491404,0.558353,0.747189,0.239598,7.378268e-02,7.545528e-02,6.629951,7.654960,0.894311,5
3,msc,2.175266,candidate,0.459916,7.159795,0.001712,268.634040,0.558190,0.746830,0.239580,5.478872e-04,5.588897e-04,6.639715,7.679148,0.894822,5
11,absorbance_snv_sg_d2,1.798196,candidate,0.110058,0.469570,0.001709,64.389573,0.370602,0.474474,0.091712,4.331172e-08,4.545338e-08,8.567889,8.474181,0.976453,3
1,absorbance,1.053985,candidate,0.114208,0.844915,0.004926,23.182776,0.349637,0.429000,0.035546,3.567313e-03,3.681745e-03,7.278988,8.571290,0.995863,1
10,absorbance_snv_sg_d1,0.447945,candidate,0.157345,4.837499,0.002351,66.921961,0.441216,0.659446,0.231424,3.065252e-05,3.581067e-05,7.233195,8.432383,0.966462,3
8,absorbance_sg_d1,0.223063,weak_class_separation,0.094941,1.049418,0.001106,85.845908,0.379046,0.404402,0.280832,9.192243e-07,1.099081e-06,7.734595,6.792165,0.965286,3
0,raw,-0.202665,weak_class_separation,0.094625,1.048966,0.005812,16.282363,0.359280,0.399048,0.087792,1.085546e-03,1.303136e-03,7.841042,10.603457,0.997538,1


In [28]:
best_5_pca_mean = pca_comparison_df_mean_scored[pca_comparison_df_mean_scored['selection_flag']=='candidate'].sort_values('class_trace_ratio', ascending=False).head(5)

In [29]:
plot_pca_metric_tradeoff(
    best_5_pca_mean,
    x_metric="batch_trace_ratio",
    y_metric="class_trace_ratio",
    color_by="preprocessing",
    symbol_by="matrix_method",
    title="Object mean PCA — class separation vs batch effect",
)

plot_pca_metric_ranking(
    best_5_pca_mean,
    metric="selection_score",
    ascending=False,
    title="Object mean PCA — preprocessing ranking",
)

plot_pca_metric_ranking(
    best_5_pca_mean,
    metric="projection_train_q_ratio",
    ascending=True,
    title="Object mean PCA — projection/train Q ratio",
)

In [30]:
for name in best_5_pca_mean["preprocessing"].to_list():
    res = pca_results_mean[name]
    item = preprocessed_data_mean[name]

    Ttr = res["T_train"]
    Tpr = res["T_projection"]

    T_all = np.vstack([Ttr, Tpr])

    y_all = np.concatenate([
        item["y_train"],
        item["y_projection"],
    ])

    subset_all = np.array(
        ["train"] * Ttr.shape[0]
        + ["projection"] * Tpr.shape[0]
    )

    meta_train_tmp = (
        pd.DataFrame(item["meta_train"])
        if isinstance(item["meta_train"], dict)
        else item["meta_train"].copy()
    )

    meta_projection_tmp = (
        pd.DataFrame(item["meta_projection"])
        if isinstance(item["meta_projection"], dict)
        else item["meta_projection"].copy()
    )

    meta_all_tmp = pd.concat(
        [
            meta_train_tmp.assign(subset="train"),
            meta_projection_tmp.assign(subset="projection"),
        ],
        ignore_index=True,
    )

    plot_scores(
        T_all,
        dims=(1, 2),

        # couleur = type de noix
        labels=y_all,
        color_by="label",

        # forme = batch
        batches=meta_all_tmp["batch"],
        symbol_by="batch",

        # rempli / ouvert = set
        subset=meta_all_tmp["subset"],
        contour_by="subset",

        object_ids=meta_all_tmp["object_id"],
        source_images=meta_all_tmp["source_image"],

        title=f"PCA scores — nut color, batch symbol, set contour — {name}",
    )

## all pixels

In [31]:
pca_results_pixel = {}

for key, item in preprocessed_data_pixel.items():
    Xtr = item["X_train"]
    Xpr = item["X_projection"]
    ytr = item["y_train"]
    ypr = item["y_projection"]
    
    pca_model = PCAModel(
        n_components=n_components,
        center=True,
    )
    
    pca_model.fit(Xtr)
    
    Ttr = pca_model.transform(Xtr)
    Tpr = pca_model.transform(Xpr)
    
    # Reconstruction with the first n_components_diagnostic components
    Tpr_A = Tpr[:, :n_components_diagnostic]
    P_A = pca_model.loadings_[:, :n_components_diagnostic]
    Xpr_hat = Tpr_A @ P_A.T + pca_model.mean_
    
    rmse_pr = np.sqrt(np.mean((Xpr - Xpr_hat) ** 2, axis=1))
    
    # Class separation scores on training scores
    sep = binary_class_separation_scores(
        Ttr,
        ytr,
        n_components=n_components_diagnostic,
    )
    
    pca_results_pixel[key] = {
        "pca": pca_model,
        "T_train": Ttr,
        "T_projection": Tpr,
        "rmse_projection": rmse_pr,
        "separation": sep,
        "explained_variance_ratio": pca_model.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": pca_model.cumulative_explained_variance_ratio_,
    }

print("PCA models fitted for:")
for name in pca_results_pixel:
    print("-", name)

PCA models fitted for:
- raw
- absorbance
- snv
- msc
- 1st_derivative
- 2nd_derivative
- absorbance_snv
- absorbance_msc
- absorbance_sg_d1
- absorbance_sg_d2
- absorbance_snv_sg_d1
- absorbance_snv_sg_d2


In [32]:
pca_comparison_df_pixel = build_pca_metrics_dataframe(
    pca_results=pca_results_pixel,
    preprocessed_data=preprocessed_data_pixel,
    matrix_method="all_pixels",
    candidate_preprocessings=list(pca_results_pixel.keys()),
    n_components_metrics=3,
)
pca_comparison_df_pixel["projection_q_deviation"] = np.abs(
    np.log(pca_comparison_df_pixel["projection_train_q_ratio"].astype(float))
)

pca_comparison_df_pixel_scored = add_pca_selection_score(
    pca_comparison_df_pixel,
    profile="pixel",
    group_col=None,
    score_col="selection_score",
)

display(
    pca_comparison_df_pixel_scored[[
        "preprocessing","selection_score", "selection_flag", "class_trace_ratio", "batch_trace_ratio", "class_over_batch_ratio",
        "object_class_trace_ratio", "object_batch_trace_ratio", "object_over_intra_ratio", "mean_intra_object_trace", 
        "mean_train_projection_shift_norm", "max_train_projection_shift_norm", "projection_q_deviation", 
        "train_q_q95", "projection_q_q95", "train_t2_q95", "projection_t2_q95", "cum_pc3", "ncomp_95",
    ]]
    .sort_values("selection_score", ascending=False)
)

,preprocessing,selection_score,selection_flag,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace,mean_train_projection_shift_norm,max_train_projection_shift_norm,projection_q_deviation,train_q_q95,projection_q_q95,train_t2_q95,projection_t2_q95,cum_pc3,ncomp_95
2,snv,12.605493,candidate,0.023144,0.000248,93.470731,0.290103,0.000316,0.122525,1.183047e+00,0.165701,0.206500,0.035676,8.564249e-01,8.218108e-01,8.795986,8.320018,0.777391,12
6,absorbance_snv,7.992449,candidate,0.015389,0.000293,52.601586,0.181360,0.000710,0.105817,1.507204e+00,0.147065,0.230697,0.035374,1.036629e+00,1.019988e+00,8.788042,8.644711,0.783761,8
0,raw,1.478076,batch_sensitive,0.018367,0.002193,8.373756,0.094324,0.005822,0.363777,5.598002e-01,0.171786,0.195702,0.100634,6.526538e-03,7.908120e-03,8.528279,8.951208,0.996359,1
10,absorbance_snv_sg_d1,0.712151,candidate,0.008690,0.000371,23.453734,0.072670,0.000591,0.144236,2.057566e-03,0.163859,0.266455,0.086250,2.040722e-04,1.900184e-04,9.667165,9.589635,0.962033,3
8,absorbance_sg_d1,0.085159,unstable_projection,0.002194,0.000021,106.546387,0.069350,0.000073,0.045141,4.531268e-04,0.055363,0.057530,1.402993,3.670633e-06,4.163916e-06,6.852156,6.489380,0.996724,1
11,absorbance_snv_sg_d2,-1.151743,weak_object_separation,0.003028,0.000225,13.435885,0.045736,0.000112,0.071694,1.108855e-05,0.100214,0.147448,0.022842,3.803042e-07,3.601833e-07,10.328366,9.477376,0.986041,1
5,2nd_derivative,-2.782053,batch_sensitive,0.006658,0.001248,5.334391,0.057718,0.008214,0.153357,2.551983e-08,0.065276,0.098384,0.104624,1.786859e-09,2.171449e-09,7.822091,8.132212,0.972983,3
9,absorbance_sg_d2,-2.800497,weak_object_separation,0.000863,0.000014,59.563666,0.042162,0.000024,0.029906,2.677698e-06,0.032312,0.033982,2.042491,5.910030e-09,6.187972e-09,6.708432,5.792810,0.999141,1
4,1st_derivative,-3.422913,weak_object_separation,0.008064,0.002258,3.571194,0.047963,0.009794,0.237835,1.085323e-05,0.089448,0.105957,0.092757,1.532455e-06,1.870731e-06,8.158547,9.553277,0.955524,3
1,absorbance,-6.105593,unstable_projection,0.001731,0.000086,20.206943,0.067722,0.000838,0.047119,9.446355e+00,0.039600,0.052196,2.053867,2.698556e-02,2.741392e-02,4.418402,4.337148,0.999181,1


In [33]:
best_5_pca_pixel = pca_comparison_df_pixel_scored[pca_comparison_df_pixel_scored['selection_flag'] == 'candidate'].sort_values('object_class_trace_ratio', ascending=False).head(5)

In [34]:
for name in best_5_pca_pixel["preprocessing"].to_list():
    if name not in pca_results_pixel:
        continue

    res = pca_results_pixel[name]
    item = preprocessed_data_pixel[name]

    Ttr = res["T_train"]
    Tpr = res["T_projection"]

    T_all = np.vstack([Ttr, Tpr])

    y_all = np.concatenate([
        item["y_train"],
        item["y_projection"],
    ])

    subset_all = np.array(
        ["train"] * Ttr.shape[0]
        + ["projection"] * Tpr.shape[0]
    )

    meta_train_tmp = (
        pd.DataFrame(item["meta_train"])
        if isinstance(item["meta_train"], dict)
        else item["meta_train"].copy()
    )

    meta_projection_tmp = (
        pd.DataFrame(item["meta_projection"])
        if isinstance(item["meta_projection"], dict)
        else item["meta_projection"].copy()
    )

    meta_all_tmp = pd.concat(
        [
            meta_train_tmp.assign(subset="train"),
            meta_projection_tmp.assign(subset="projection"),
        ],
        ignore_index=True,
    )

    df_scores_pixel = build_scores_dataframe(
        scores=T_all,
        labels=y_all,
        meta=meta_all_tmp,
        subset=subset_all,
        dims=(1, 2, 3),
        score_prefix="C",
    )

    # 1. Scatter sous-échantillonné
    df_sample = sample_scores_dataframe(
        df_scores_pixel,
        group_cols=["label", "subset", "batch"],
        n_per_group=300,
    )

    plot_scores(
        df_sample[["C1", "C2", "C3"]].values,
        dims=(1, 2),
        labels=df_sample["label"],
        color_by="label",
        batches=df_sample["batch"],
        symbol_by="batch",
        subset=df_sample["subset"],
        contour_by="subset",
        object_ids=df_sample["object_id"],
        source_images=df_sample["source_image"],
        title=f"All pixels PCA — sampled scores — {name}",
    )

    # # 2. Densité pixel
    # plot_scores_density(
    #     df_scores_pixel,
    #     x="C1",
    #     y="C2",
    #     color_by="label",
    #     facet_col="subset",
    #     mode="contour",
    #     title=f"All pixels PCA — score density — {name}",
    # )

    # # 3. Distribution PC1
    # plot_scores_distribution(
    #     df_scores_pixel,
    #     score_col="C1",
    #     x_by="label",
    #     color_by="label",
    #     facet_col="subset",
    #     kind="violin",
    #     title=f"All pixels PCA — C1 distribution — {name}",
    # )

    # # 4. Distribution PC2
    # plot_scores_distribution(
    #     df_scores_pixel,
    #     score_col="C2",
    #     x_by="label",
    #     color_by="label",
    #     facet_col="subset",
    #     kind="violin",
    #     title=f"All pixels PCA — C2 distribution — {name}",
    # )

    # 5. Résumé objet à partir des pixels
    df_object_scores = summarize_scores_by_object(
        df_scores_pixel,
        score_cols=("C1", "C2", "C3"),
        object_col="object_id",
    )

    plot_object_score_summary(
        df_object_scores,
        x="C1_mean",
        y="C2_mean",
        color_by="label",
        symbol_by="batch",
        facet_col="subset",
        title=f"All pixels PCA — object-level mean scores — {name}",
    )

## balanced pixels

In [35]:
pca_results_balanced_pixel = {}

for key, item in preprocessed_data_balanced_pixel.items():
    Xtr = item["X_train"]
    Xpr = item["X_projection"]
    ytr = item["y_train"]
    ypr = item["y_projection"]
    
    pca_model = PCAModel(
        n_components=n_components,
        center=True,
    )
    
    pca_model.fit(Xtr)
    
    Ttr = pca_model.transform(Xtr)
    Tpr = pca_model.transform(Xpr)
    
    # Reconstruction with the first n_components_diagnostic components
    Tpr_A = Tpr[:, :n_components_diagnostic]
    P_A = pca_model.loadings_[:, :n_components_diagnostic]
    Xpr_hat = Tpr_A @ P_A.T + pca_model.mean_
    
    rmse_pr = np.sqrt(np.mean((Xpr - Xpr_hat) ** 2, axis=1))
    
    # Class separation scores on training scores
    sep = binary_class_separation_scores(
        Ttr,
        ytr,
        n_components=n_components_diagnostic,
    )
    
    pca_results_balanced_pixel[key] = {
        "pca": pca_model,
        "T_train": Ttr,
        "T_projection": Tpr,
        "rmse_projection": rmse_pr,
        "separation": sep,
        "explained_variance_ratio": pca_model.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": pca_model.cumulative_explained_variance_ratio_,
    }

print("PCA models fitted for:")
for name in pca_results_balanced_pixel:
    print("-", name)

PCA models fitted for:
- raw
- absorbance
- snv
- msc
- 1st_derivative
- 2nd_derivative
- absorbance_snv
- absorbance_msc
- absorbance_sg_d1
- absorbance_sg_d2
- absorbance_snv_sg_d1
- absorbance_snv_sg_d2


In [37]:
pca_comparison_df_balanced_pixel = build_pca_metrics_dataframe(
    pca_results=pca_results_balanced_pixel,
    preprocessed_data=preprocessed_data_balanced_pixel,
    matrix_method="balanced_pixels",
    candidate_preprocessings=list(pca_results_balanced_pixel.keys()),
    n_components_metrics=3,
)
pca_comparison_df_balanced_pixel["projection_q_deviation"] = np.abs(
    np.log(pca_comparison_df_balanced_pixel["projection_train_q_ratio"].astype(float))
)

pca_comparison_df_balanced_pixel_scored = add_pca_selection_score(
    pca_comparison_df_balanced_pixel,
    profile="pixel",
    group_col=None,
    score_col="selection_score",
)

display(
    pca_comparison_df_balanced_pixel_scored[[
        "preprocessing", "selection_score", "selection_flag", "class_trace_ratio", "batch_trace_ratio", "class_over_batch_ratio",
        "object_class_trace_ratio", "object_batch_trace_ratio", "object_over_intra_ratio", "mean_intra_object_trace",
        "mean_train_projection_shift_norm", "max_train_projection_shift_norm", "projection_q_deviation", "train_q_q95", "projection_q_q95",
        "train_t2_q95", "projection_t2_q95", "cum_pc3", "ncomp_95",
    ]]
    .sort_values("selection_score", ascending=False)
)

,preprocessing,selection_score,selection_flag,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace,mean_train_projection_shift_norm,max_train_projection_shift_norm,projection_q_deviation,train_q_q95,projection_q_q95,train_t2_q95,projection_t2_q95,cum_pc3,ncomp_95
2,snv,10.065900,candidate,0.024467,1.031776e-04,237.131675,0.285004,0.000332,0.123525,1.210896e+00,0.145923,0.181014,0.048364,9.210261e-01,8.897052e-01,8.544435,8.126501,0.787371,11
6,absorbance_snv,5.771306,candidate,0.013800,1.311928e-04,105.188308,0.160438,0.000754,0.112641,1.536458e+00,0.131272,0.203127,0.013282,1.092736e+00,1.040893e+00,8.621341,8.482665,0.790677,8
0,raw,1.611250,batch_sensitive,0.027413,1.712254e-03,16.010066,0.101886,0.006826,0.348487,5.660036e-01,0.197002,0.200990,0.061533,6.615025e-03,7.566354e-03,8.638193,8.973291,0.996192,1
10,absorbance_snv_sg_d1,1.117115,candidate,0.007944,1.147039e-04,69.252238,0.069100,0.000271,0.155650,2.044967e-03,0.150207,0.236769,0.116036,2.099872e-04,1.919280e-04,9.599100,9.252269,0.961936,3
8,absorbance_sg_d1,-0.326624,unstable_projection,0.002658,1.568856e-06,1694.332237,0.052041,0.000045,0.053889,4.696811e-04,0.055279,0.056918,1.126579,3.749862e-06,4.018151e-06,6.970344,6.412067,0.997002,1
11,absorbance_snv_sg_d2,-0.753845,weak_object_separation,0.002573,1.622272e-04,15.858151,0.040633,0.000805,0.081492,1.103585e-05,0.092365,0.127962,0.072386,3.858157e-07,3.519007e-07,10.360421,9.386147,0.986282,1
5,2nd_derivative,-1.064328,batch_sensitive,0.008031,9.196663e-04,8.732375,0.060176,0.006730,0.153773,2.528270e-08,0.068481,0.110176,0.056057,1.718975e-09,2.026935e-09,7.851160,8.100881,0.972483,3
4,1st_derivative,-1.775579,batch_sensitive,0.009423,1.768427e-03,5.328306,0.050711,0.008864,0.233136,1.082125e-05,0.092030,0.092710,0.036709,1.493222e-06,1.714989e-06,8.191724,9.436274,0.953395,3
9,absorbance_sg_d2,-2.591201,weak_object_separation,0.001068,7.450660e-07,1432.799242,0.027100,0.000025,0.038880,2.776298e-06,0.031983,0.034187,1.737616,5.921966e-09,5.974211e-09,6.758558,5.738543,0.999220,1
7,absorbance_msc,-5.785278,weak_object_separation,0.000139,1.151062e-04,1.207528,0.005332,0.004315,0.026020,8.631252e-01,0.011037,0.016965,0.617996,5.417220e-02,5.545246e-02,0.289253,0.269775,0.965912,3


In [38]:
best_5_pca_balanced_pixel = pca_comparison_df_balanced_pixel_scored[pca_comparison_df_balanced_pixel_scored['selection_flag'] == 'candidate'].sort_values('object_class_trace_ratio', ascending=False).head(5)

In [39]:
for name in best_5_pca_balanced_pixel['preprocessing'].to_list():

    res = pca_results_balanced_pixel[name]
    item = preprocessed_data_balanced_pixel[name]

    Ttr = res["T_train"]
    Tpr = res["T_projection"]

    T_all = np.vstack([Ttr, Tpr])

    y_all = np.concatenate([
        item["y_train"],
        item["y_projection"],
    ])

    subset_all = np.array(
        ["train"] * Ttr.shape[0]
        + ["projection"] * Tpr.shape[0]
    )

    meta_train_tmp = (
        pd.DataFrame(item["meta_train"])
        if isinstance(item["meta_train"], dict)
        else item["meta_train"].copy()
    )

    meta_projection_tmp = (
        pd.DataFrame(item["meta_projection"])
        if isinstance(item["meta_projection"], dict)
        else item["meta_projection"].copy()
    )

    meta_all_tmp = pd.concat(
        [
            meta_train_tmp.assign(subset="train"),
            meta_projection_tmp.assign(subset="projection"),
        ],
        ignore_index=True,
    )

    df_scores_balanced_pixel = build_scores_dataframe(
        scores=T_all,
        labels=y_all,
        meta=meta_all_tmp,
        subset=subset_all,
        dims=(1, 2, 3),
        score_prefix="C",
    )

    # 1. Scatter sous-échantillonné
    df_sample = sample_scores_dataframe(
        df_scores_balanced_pixel,
        group_cols=["label", "subset", "batch"],
        n_per_group=500,
    )

    plot_scores(
        df_sample[["C1", "C2", "C3"]].values,
        dims=(1, 2),
        labels=df_sample["label"],
        color_by="label",
        batches=df_sample["batch"],
        symbol_by="batch",
        subset=df_sample["subset"],
        contour_by="subset",
        object_ids=df_sample["object_id"],
        source_images=df_sample["source_image"],
        title=f"Balanced pixels PCA — sampled scores — {name}",
    )

    # # 2. Densité pixel
    # plot_scores_density(
    #     df_scores_balanced_pixel,
    #     x="C1",
    #     y="C2",
    #     color_by="label",
    #     facet_col="subset",
    #     mode="contour",
    #     title=f"Balanced pixels PCA — score density — {name}",
    # )

    # # 3. Distribution PC1
    # plot_scores_distribution(
    #     df_scores_balanced_pixel,
    #     score_col="C1",
    #     x_by="label",
    #     color_by="label",
    #     facet_col="subset",
    #     kind="violin",
    #     title=f"Balanced pixels PCA — C1 distribution — {name}",
    # )

    # 4. Résumé objet
    df_object_scores = summarize_scores_by_object(
        df_scores_balanced_pixel,
        score_cols=("C1", "C2", "C3"),
        object_col="object_id",
    )

    plot_object_score_summary(
        df_object_scores,
        x="C1_mean",
        y="C2_mean",
        color_by="label",
        symbol_by="batch",
        facet_col="subset",
        title=f"Balanced pixels PCA — object-level mean scores — {name}",
    )

## global comparison

In [42]:
best_pca_comparison_all_df = pd.concat(
    [
        best_5_pca_mean,
        best_5_pca_pixel,
        best_5_pca_balanced_pixel,
    ],
    ignore_index=True,
)

display(
    best_pca_comparison_all_df[[
    # Identification
    "matrix_method",
    "preprocessing",
    "selection_score",
    "selection_flag",

    # Main class separation
    "class_trace_ratio",
    "object_class_trace_ratio",

    # Batch effect
    "batch_trace_ratio",
    "object_batch_trace_ratio",

    # Class / batch compromise
    "class_over_batch_ratio",

    # Pixel-object structure
    "object_over_intra_ratio",
    "mean_intra_object_trace",

    # Projection stability
    "mean_train_projection_shift_norm",
    "max_train_projection_shift_norm",
    "projection_q_deviation",

    # PCA compactness
    "cum_pc3",
    "ncomp_95",

    # Optional diagnostic
    "mahalanobis_pc1_pc2_pc3",
    ]]
    .sort_values("selection_score", ascending=False)
)

,matrix_method,preprocessing,selection_score,selection_flag,class_trace_ratio,object_class_trace_ratio,batch_trace_ratio,object_batch_trace_ratio,class_over_batch_ratio,object_over_intra_ratio,mean_intra_object_trace,mean_train_projection_shift_norm,max_train_projection_shift_norm,projection_q_deviation,cum_pc3,ncomp_95,mahalanobis_pc1_pc2_pc3
5,all_pixels,snv,12.605493,candidate,0.023144,0.290103,0.000248,0.000316,93.470731,0.122525,1.183047,0.165701,0.206500,0.035676,0.777391,12,0.802725
8,balanced_pixels,snv,10.065900,candidate,0.024467,0.285004,0.000103,0.000332,237.131675,0.123525,1.210896,0.145923,0.181014,0.048364,0.787371,11,0.850117
6,all_pixels,absorbance_snv,7.992449,candidate,0.015389,0.181360,0.000293,0.000710,52.601586,0.105817,1.507204,0.147065,0.230697,0.035374,0.783761,8,0.582290
9,balanced_pixels,absorbance_snv,5.771306,candidate,0.013800,0.160438,0.000131,0.000754,105.188308,0.112641,1.536458,0.131272,0.203127,0.013282,0.790677,8,0.552258
1,object_mean,absorbance_msc,2.273695,candidate,0.480243,NaN,0.002677,NaN,179.402809,NaN,NaN,0.493531,0.784386,0.244209,0.899133,5,6.627051
0,object_mean,absorbance_snv,2.273591,candidate,0.481741,NaN,0.002691,NaN,179.048812,NaN,NaN,0.493864,0.785219,0.244553,0.898381,5,6.681698
2,object_mean,snv,2.191771,candidate,0.460763,NaN,0.001716,NaN,268.491404,NaN,NaN,0.558353,0.747189,0.239598,0.894311,5,7.332611
3,object_mean,msc,2.175266,candidate,0.459916,NaN,0.001712,NaN,268.634040,NaN,NaN,0.558190,0.746830,0.239580,0.894822,5,7.159795
10,balanced_pixels,absorbance_snv_sg_d1,1.117115,candidate,0.007944,0.069100,0.000115,0.000271,69.252238,0.155650,0.002045,0.150207,0.236769,0.116036,0.961936,3,0.365100
7,all_pixels,absorbance_snv_sg_d1,0.712151,candidate,0.008690,0.072670,0.000371,0.000591,23.453734,0.144236,0.002058,0.163859,0.266455,0.086250,0.962033,3,0.362713


In [43]:
plot_pca_metric_heatmap(
    best_pca_comparison_all_df,
    metric="selection_score",
    index_col="preprocessing",
    column_col="matrix_method",
    title="Global PCA selection score",
)

plot_pca_metric_heatmap(
    best_pca_comparison_all_df,
    metric="class_trace_ratio",
    index_col="preprocessing",
    column_col="matrix_method",
    title="Class separation by preprocessing and representation",
)

plot_pca_metric_heatmap(
    best_pca_comparison_all_df,
    metric="batch_trace_ratio",
    index_col="preprocessing",
    column_col="matrix_method",
    title="Batch effect by preprocessing and representation",
)

plot_pca_metric_tradeoff(
    best_pca_comparison_all_df,
    x_metric="batch_trace_ratio",
    y_metric="class_trace_ratio",
    color_by="matrix_method",
    symbol_by="preprocessing",
    title="PCA trade-off — class separation vs batch effect",
)

plot_pca_metric_ranking(
    best_pca_comparison_all_df,
    metric="selection_score",
    group_col="matrix_method",
    label_col="preprocessing",
    ascending=False,
    title="PCA preprocessing ranking by representation",
)